# Comparación de modelos fundacionales de Time Series. Cambios respecto a la prueba 2:
Implementación de los modelos fundacionales que no estan originalmente en timecopilot
Los pruebo

**1. Instalo paquetes**

In [0]:
%pip install --upgrade black fsspec
%pip install --no-deps "momentfm==0.1.4"
%pip install --no-deps "git+https://github.com/hitachi-ais/citras-fm.git@v3.0.0"
%pip install --no-deps "tabicl[forecast]==2.0.3"

In [0]:
#instalo las cositas
%pip install --no-deps -e /Workspace/Users/anamaria.afonso@practicas.repsol.com/timecopilot_pruebas
# dependencias de timecopilot que no se instalan por usar --no-deps
%pip install "statsforecast>=2.0.2" "catboost>=1.2.10" fire "logfire>=4.11.0" "openai>=1.99.7" "opentelemetry-instrumentation>=0.58b0" "peft>=0.20.0,<1" "plotly>=6.3.1" "pydantic-ai>=0.7.0" "xgboost>=3.2.0"
%pip install "pydantic-ai-slim[google]"
%pip install tsfeatures
%pip install foundationforecast
%pip install mlforecast
%pip install neuralforecast
%pip install prophet

In [0]:
%pip install "transformers==4.57.6"

In [0]:
dbutils.library.restartPython()

**2. Importo mis librerias**

In [0]:
#importo librerias necesarias
import timecopilot
import os
import sys
import pandas as pd
import nest_asyncio
#from timecopilot import TimeCopilot --> no lo uso, no lo necesito para el pipeline determinista
nest_asyncio.apply()
from timecopilot import TimeCopilotForecaster
#importo los modelos que quiero usar
#incluidos en timecopilot
from timecopilot.models.foundation.chronos import Chronos
from timecopilot.models.foundation.tirex import TiRex
from timecopilot.models.foundation import Moirai
from timecopilot.models.foundation import Timestral
from timecopilot.models.foundation.timesfm import TimesFM
from timecopilot.models.foundation.toto import Toto
from timecopilot.models.foundation.patchtst_fm import PatchTSTFM
from timecopilot.models.foundation.flowstate import FlowState
from timecopilot.models.foundation.t0 import T0
from tabpfn_time_series import TabPFNMode
from timecopilot.models.foundation.tabpfn import TABPFN_V2_MODEL, TABPFN_V3_MODEL, TabPFN
from timecopilot.models.foundation.timegpt import TimeGPT
from timecopilot.models.foundation.timeh2o import TimeH2O
from timecopilot.models.stats import AutoARIMA, SeasonalNaive, AutoETS
from timecopilot.models.ml import AutoXGBoost, AutoLGBM, AutoRandomForest, AutoLinearRegression, AutoRidge, AutoLasso
#agregados por mi, confirmado que ya funcionan
import sys
sys.path.insert(0, "/Workspace/Users/anamaria.afonso@practicas.repsol.com/Time-MoE")
from timecopilot.models.foundation.time_moe import TimeMoE 
LAG_LLAMA_ROOT = (
    "/Workspace/Users/anamaria.afonso@practicas.repsol.com/lag-llama"
)
if LAG_LLAMA_ROOT in sys.path:
    sys.path.remove(LAG_LLAMA_ROOT)
sys.path.insert(0, LAG_LLAMA_ROOT)
from timecopilot.models.foundation.lag_llama import LagLlama
from timecopilot.models.foundation.moment import MOMENT
from timecopilot.models.foundation.citras_fm import CITRASFM
from timecopilot.models.foundation.tabicl_v2 import TabICLv2
#importo metodos de evaluacion
from functools import partial
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import (
    # Errores absolutos/cuadráticos
    mae,
    mse,
    rmse,
    # Errores porcentuales
    mape,
    smape,
    wape,
    # Errores escalados
    mase,
    msse,
    rmsse,
    # Sesgo
    bias,
    cfe,
    # Métricas de inventario
    pis,
    spis,
    # Probabilísticas
    coverage,
    calibration,
    winkler_score,
    scaled_crps,
)

from functools import partial

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from plotly.colors import qualitative

**3. Importo mis APIs**

In [0]:
import os

os.environ["HF_TOKEN"] = dbutils.secrets.get(
    scope="timecopilot",
    key="hf-token",
)
os.environ["NIXTLA_API_KEY"] = dbutils.secrets.get(
    scope="timecopilot",
    key="nixtila_api_key",
)
os.environ["TABPFN_TOKEN"] = dbutils.secrets.get(
    scope="timecopilot",
    key="tabpfn_token",
)

**4. Importo mi DF**

explicación del formato del df:
df: unique_id | ds | y
X_df: unique_id | ds | exog1 | exog2 | ... | exogN

Timecopilot espera este formato siempre, puedo renombrar mis datos antes de llamar a la librería:\
df = df.rename(columns={
    "id_serie": "unique_id",
    "fecha": "ds",
    "valor": "y",
})

In [0]:
DATA = "https://timecopilot.s3.amazonaws.com/public/data/electricity_price"
df = pd.read_parquet(f"{DATA}/train.parquet")
X_df = pd.read_parquet(f"{DATA}/futr_exog.parquet")

df["ds"] = pd.to_datetime(df["ds"])
X_df["ds"] = pd.to_datetime(X_df["ds"])

print("df:", df.shape, df.columns.tolist())
print("X_df:", X_df.shape, X_df.columns.tolist())
print("y en X_df:", "y" in X_df.columns)

display(df.head())
display(X_df.head())
display(
    X_df.groupby("unique_id").size()
        .rename("n_futuras").reset_index()
)
TimeCopilotForecaster.plot(df)

**5. Creo mi Time Copilot Forecaster**

In [0]:
"""context_length = 2048
batch_size = 32

chronos_av_models = [
    "amazon/chronos-2",
    "amazon/chronos-bolt-tiny",
    "amazon/chronos-bolt-mini",
    "amazon/chronos-bolt-small",
    "amazon/chronos-bolt-base",
    "amazon/chronos-t5-tiny",
    "amazon/chronos-t5-mini",
    "amazon/chronos-t5-small",
    "amazon/chronos-t5-base",
    "amazon/chronos-t5-large",
]

chronos_models = [Chronos(repo_id=model, alias=model) for model in chronos_av_models]
tcf = TimeCopilotForecaster(
        models=[
            TimeMoE(
                repo_id="Maple728/TimeMoE-50M",
                alias="TimeMoE-50M",
                context_length=512,
                seasonality=12,
                calibration_windows=5,
                calibration_step_size=1,
            ),
            TiRex(repo_id="NX-AI/TiRex", alias="TiRex"),
            Moirai(repo_id="Salesforce/moirai-1.0-R-small", alias="Moirai"),
            LagLlama(alias="Lag-Llama", num_samples=100),
            MOMENT(alias="MOMENT", calibration_windows=3),
            CITRASFM(),
            TabICLv2(),
            PatchTSTFM(
                repo_id="ibm-research/patchtst-fm-r1",
                alias="PatchTST-FM-r1",
                context_length=context_length,
                batch_size=batch_size,
            ),
            PatchTSTFM(
                repo_id="ibm-granite/granite-timeseries-patchtst-fm-r1",
                alias="Granite-PatchTST-FM-r1",
                context_length=context_length,
                batch_size=batch_size,
            ),
            PatchTSTFM(
                repo_id="ibm-granite/granite-timeseries-patchtst-fm-r2",
                alias="Granite-PatchTST-FM-r2",
                context_length=context_length,
                batch_size=batch_size,
            ),
            AutoARIMA(),
            SeasonalNaive(),
            AutoETS(),
            AutoXGBoost(cv_n_windows=2),
            AutoLGBM(cv_n_windows=2),
            AutoRandomForest(cv_n_windows=2),
            AutoLinearRegression(cv_n_windows=2),
            AutoRidge(cv_n_windows=2),
            AutoLasso(cv_n_windows=2),
            *chronos_models,
            TimeGPT(
                alias="TimeGPT"
            ),
            Timestral(repo_id="mistralai/Timestral-5M", alias="Timestral-5M", context_length=512, batch_size=2),
            Timestral(repo_id="mistralai/Timestral-28M", alias="Timestral-28M", context_length=512, batch_size=2),
            Timestral(repo_id="mistralai/Timestral-123M", alias="Timestral-123M", context_length=512, batch_size=2),
            Timestral(repo_id="mistralai/Timestral-209M", alias="Timestral-209M", context_length=512, batch_size=2),
            TimesFM(
                repo_id="google/timesfm-1.0-200m-pytorch",
                alias="TimesFM-1.0",
                context_length=context_length,
            ),
            TimesFM(
                repo_id="google/timesfm-2.0-500m-pytorch",
                alias="TimesFM-2.0",
                context_length=context_length,
            ),
            TimesFM(
                repo_id="google/timesfm-2.5-200m-pytorch",
                alias="TimesFM-2.5",
                context_length=context_length,
            ),
            TimesFM(
                repo_id="google/timesfm-3.0-pytorch",
                alias="TimesFM-3",
                context_length=context_length,
            ),
            Toto(repo_id="Datadog/Toto-Open-Base-1.0", alias="Toto-1-151M"),
            Toto(repo_id="Datadog/Toto-2.0-4m", alias="Toto-2-4M"),
            Toto(repo_id="Datadog/Toto-2.0-22m", alias="Toto-2-22M"),
            FlowState(),
            T0(repo_id="theforecastingcompany/t0-alpha", alias="t0-alpha"),
            T0(repo_id="theforecastingcompany/t0-beta", alias="t0-beta"),
            T0(repo_id="theforecastingcompany/t0-gamma", alias="t0-gamma"),
            TabPFN(
                model_path=TABPFN_V2_MODEL,
                mode=TabPFNMode.LOCAL,
                alias="TabPFN-2",
            ),
            TabPFN(
                model_path=TABPFN_V3_MODEL,
                mode=TabPFNMode.LOCAL,
                context_length=32768,
                alias="TabPFN-3",
            ),
            TimeH2O(repo_id="h2oai/timeh2o-v1", alias="timeh2o-v1"),
            TimeH2O(repo_id="h2oai/timeh2o-v1-tiny", alias="timeh2o-v1-tiny"),
        ]
)"""

**6. Genero el forecast y lo guardo**

In [0]:
# Modelos que utilizarán las variables exógenas --> ojo! estos modelos no son necesariamente los únicos que de manera nativa usan las exógenas --> son los que tienen el wrapper adaptado para usar exógenas en timecopilot
context_length = 2048
batch_size = 32

models_with_exog = [
    Chronos(
        repo_id="amazon/chronos-2",
        alias="amazon/chronos-2",
    ),
    TimeGPT(alias="TimeGPT"),
    TabICLv2(),
    TabPFN(
        model_path=TABPFN_V2_MODEL,
        mode=TabPFNMode.LOCAL,
        context_length=1000, #para que se ejecute en local y no en CPU 
        alias="TabPFN-2",
    ),
    TabPFN(
        model_path=TABPFN_V3_MODEL,
        mode=TabPFNMode.LOCAL,
        context_length=1000, #para que se ejecute en local y no en CPU, tarda mas pero no necesito API de cliente 
        alias="TabPFN-3",
    ),
    Timestral(
        repo_id="mistralai/Timestral-5M",
        alias="Timestral-5M",
        context_length=512,
        batch_size=2,
    ),
    Timestral(
        repo_id="mistralai/Timestral-28M",
        alias="Timestral-28M",
        context_length=512,
        batch_size=2,
    ),
    Timestral(
        repo_id="mistralai/Timestral-123M",
        alias="Timestral-123M",
        context_length=512,
        batch_size=2,
    ),
    Timestral(
        repo_id="mistralai/Timestral-209M",
        alias="Timestral-209M",
        context_length=512,
        batch_size=2,
    ),
    TimesFM(
        repo_id="google/timesfm-3.0-pytorch",
        alias="TimesFM-3",
        context_length=context_length,
    ),
    T0(
        repo_id="theforecastingcompany/t0-alpha",
        alias="t0-alpha",
    ),
    T0(
        repo_id="theforecastingcompany/t0-beta",
        alias="t0-beta",
    ),
    T0(
        repo_id="theforecastingcompany/t0-gamma",
        alias="t0-gamma",
    ),
    TimeH2O(
        repo_id="h2oai/timeh2o-v1",
        alias="timeh2o-v1",
    ),
    CITRASFM(),
    Toto(
        repo_id="Datadog/Toto-Open-Base-1.0",
        alias="Toto-1-151M",
    ),
    TiRex(
        repo_id="NX-AI/TiRex-2",
        alias="TiRex-2",
        batch_size=16,
    ),
    Moirai(
        repo_id="Salesforce/moirai-1.0-R-small",
        alias="Moirai",
    ),
]

# Modelos que se ejecutarán sin variables exógenas
models_without_exog = [
    TiRex(repo_id="NX-AI/TiRex", alias="TiRex"),
    TimeMoE(
        repo_id="Maple728/TimeMoE-50M",
        alias="TimeMoE-50M",
        context_length=512,
        seasonality=12,
        calibration_windows=5,
        calibration_step_size=1,
    ),
    LagLlama(alias="Lag-Llama", num_samples=100),
    MOMENT(
        alias="MOMENT",
        calibration_windows=3,
        max_train_windows_per_series=64,
        train_stride=12,
    ), 
    PatchTSTFM(
        repo_id="ibm-research/patchtst-fm-r1",
        alias="PatchTST-FM-r1",
        context_length=context_length,
        batch_size=batch_size,
    ),
    PatchTSTFM(
        repo_id="ibm-granite/granite-timeseries-patchtst-fm-r1",
        alias="Granite-PatchTST-FM-r1",
        context_length=context_length,
        batch_size=batch_size,
    ),
    PatchTSTFM(
        repo_id="ibm-granite/granite-timeseries-patchtst-fm-r2",
        alias="Granite-PatchTST-FM-r2",
        context_length=context_length,
        batch_size=batch_size,
    ),
    AutoLinearRegression(
         num_samples=10,
         cv_n_windows=2,
        ),
    AutoRidge(
        cv_n_windows=2,
         num_samples=10,
        ),
    AutoLasso(
        cv_n_windows=2,
         num_samples=10,
        ),
    AutoARIMA(
        season_length=24,
        stepwise=True,
        approximation=True,
        max_p=3,
        max_q=3,
        max_P=1,
        max_Q=1,
        max_d=1,
        max_D=1,
        nmodels=20,
        alias="AutoARIMA",
    ),
    SeasonalNaive(),
    AutoETS(),
    AutoXGBoost(cv_n_windows=2),
    AutoLGBM(cv_n_windows=2),
    *[
        Chronos(repo_id=model, alias=model)
        for model in [
            "amazon/chronos-bolt-tiny",
            "amazon/chronos-bolt-mini",
            "amazon/chronos-bolt-small",
            "amazon/chronos-bolt-base",
            "amazon/chronos-t5-tiny",
            "amazon/chronos-t5-mini",
            "amazon/chronos-t5-small",
            "amazon/chronos-t5-base",
            "amazon/chronos-t5-large",
        ]
    ],

    TimesFM(
        repo_id="google/timesfm-1.0-200m-pytorch",
        alias="TimesFM-1.0",
        context_length=context_length,
    ),
    TimesFM(
        repo_id="google/timesfm-2.0-500m-pytorch",
        alias="TimesFM-2.0",
        context_length=context_length,
    ),
    TimesFM(
        repo_id="google/timesfm-2.5-200m-pytorch",
        alias="TimesFM-2.5",
        context_length=context_length,
    ),
    Toto(
        repo_id="Datadog/Toto-2.0-4m",
        alias="Toto-2-4M",
    ),
    Toto(
        repo_id="Datadog/Toto-2.0-22m",
        alias="Toto-2-22M",
    ),
    FlowState(),
    TimeH2O(
        repo_id="h2oai/timeh2o-v1-tiny",
        alias="timeh2o-v1-tiny",
    ),
]

In [0]:
#separo los modelos fundacionales que no toman variables exogenas de los modelos de ML
ml_model_classes = (
    AutoARIMA,
    SeasonalNaive,
    AutoETS,
    AutoXGBoost,
    AutoLGBM,
    AutoLinearRegression,
    AutoRidge,
    AutoLasso,
)

models_ml = [
    model
    for model in models_without_exog
    if isinstance(model, ml_model_classes)
]

models_foundational = [
    model
    for model in models_without_exog
    if not isinstance(model, ml_model_classes)
]

print("Modelos fundacionales:")
print([model.alias for model in models_foundational])

print("\nModelos estadísticos/ML:")
print([model.alias for model in models_ml])

In [0]:
#separo los tcf porque los modelos no adaptados dan error si intento tratarlos de manera conjunta
tcf_with_exog = TimeCopilotForecaster(
    models=models_with_exog,
)

tcf_foundational = TimeCopilotForecaster(
    models=models_foundational,
    clean_cache=True,
)
tcf_ml = TimeCopilotForecaster(
    models=models_ml,
    clean_cache=True,
)

Uso el cross validation en vez de forecast porque, hasta que tenga mis datasets oficiales, lo que quiero es probar que mi manera de implementar y evaluar estan bien, para eso necesito medir las predicciones en periodos pasados y comparar su rendimiento

In [0]:
#cross validation solo para los modelos que emplean variables exógenas
exog_cols = [
    "ex_1",
    "ex_2",
]

cv_with_exog = tcf_with_exog.cross_validation(
    df=df,
    h=12,
    futr_exog_list=exog_cols,
    quantiles=[
        0.1,
        0.2,
        0.3,
        0.5,
        0.7,
        0.8,
        0.9,
    ],
)

cv_with_exog.to_parquet("/Workspace/Users/anamaria.afonso@practicas.repsol.com/timecopilot_pruebas/mis-resultados/cv_with_exog.parquet") #para guardar la prediccion y no tener que reejecutarla cada vez que se me apaga el clúster

In [0]:
"""suspect_models = [
    AutoRandomForest(
        num_samples=10,
        cv_n_windows=2,
    ),
    AutoLinearRegression(
        num_samples= 10,
        cv_n_windows= 5,
    ),

    AutoRidge(
        num_samples = 10,
        cv_n_windows = 5,
    ),

    AutoLasso(
        num_samples= 10,
        cv_n_windows = 5,
    ),
]"""

In [0]:
"""model = suspect_models[3]

print(
    "Probando:",
    getattr(model, "alias", type(model).__name__),
)

tcf_test = TimeCopilotForecaster(
    models=[model]
)"""

In [0]:
"""df_univariate = df[
    [
        "unique_id",
        "ds",
        "y",
    ]
].copy()
cv_test = tcf_test.cross_validation(
    df=df_univariate,
    h=12,
    freq="h",
    n_windows=3,
)"""

In [0]:
#cross validation solo para los modelos fundacionales que NO emplean variables exógenas
#05.30 --> parado en el cuarto 0it
df_univariate = df[
    [
        "unique_id",
        "ds",
        "y",
    ]
].copy()

cv_foundational = tcf_foundational.cross_validation(
    df=df_univariate,
    h=12,
    freq="h",
    n_windows=1,
    quantiles=[
        0.1,
        0.2,
        0.3,
        0.5,
        0.7,
        0.8,
        0.9,
    ],
)

cv_foundational.to_parquet("/Workspace/Users/anamaria.afonso@practicas.repsol.com/timecopilot_pruebas/mis-resultados/cv_foundational.parquet") #para guardar la prediccion y no tener que reejecutarla cada vez que se me apaga el clúster

In [0]:
#cross validation solo de los modelos de machine learning
cv_ml = tcf_ml.cross_validation(
    df=df_univariate,
    h=12,
    freq="h",
    n_windows=1,
    quantiles=[
        0.1,
        0.2,
        0.3,
        0.5,
        0.7,
        0.8,
        0.9,
    ],
)

cv_ml.to_parquet("/Workspace/Users/anamaria.afonso@practicas.repsol.com/timecopilot_pruebas/mis-resultados/cv_ml.parquet") #para guardar la prediccion y no tener que reejecutarla cada vez que se me apaga el clúster

In [0]:
#leo todos mis cross validation
path_foun = "/Workspace/Users/anamaria.afonso@practicas.repsol.com/timecopilot_pruebas/mis-resultados/cv_foundational.parquet"
path_exo = "/Workspace/Users/anamaria.afonso@practicas.repsol.com/timecopilot_pruebas/mis-resultados/cv_with_exog.parquet"
path_ml =  "/Workspace/Users/anamaria.afonso@practicas.repsol.com/timecopilot_pruebas/mis-resultados/cv_ml.parquet"

cv_foundational= pd.read_parquet(path_foun)
cv_ml= pd.read_parquet(path_ml)
cv_with_exog= pd.read_parquet(path_exo)

In [0]:
#uno los modelos que no reciben exogenas (fundacionales + ML)
key_cols = [
    "unique_id",
    "ds",
    "cutoff",
    "y",
]

ml_prediction_cols = [
    column
    for column in cv_ml.columns
    if column not in key_cols
]

cv_without_exog = cv_foundational.merge(
    cv_ml[
        key_cols + ml_prediction_cols
    ],
    on=key_cols,
    how="inner",
    validate="one_to_one",
)

assert len(cv_without_exog) == len(cv_foundational)
assert len(cv_without_exog) == len(cv_ml)

print(cv_without_exog.shape)
display(cv_without_exog.head(5))

In [0]:
prediction_cols_without_exog = [
    col
    for col in cv_without_exog.columns
    if col not in key_cols
]

cv_df = cv_with_exog.merge(
    cv_without_exog[
        key_cols + prediction_cols_without_exog
    ],
    on=key_cols,
    how="inner",
    validate="one_to_one",
)

In [0]:
assert len(cv_df) == len(cv_with_exog)
assert len(cv_df) == len(cv_without_exog)

In [0]:
cv_df.to_parquet("/Workspace/Users/anamaria.afonso@practicas.repsol.com/timecopilot_pruebas/mis-resultados/cv_df_exog.parquet") #para guardar la prediccion y no tener que reejecutarla cada vez que se me apaga el clúster

In [0]:
#leo la predicción guardada para no tener que reejecutarla cada vez que se me apaga el clúster
# Ruta del archivo parquet
path = "/Workspace/Users/anamaria.afonso@practicas.repsol.com/timecopilot_pruebas/mis-resultados/cv_df_exog.parquet"
# Cargo el DataFrame
cv_df = pd.read_parquet(path)

**7. Muestro los resultados**

In [0]:
cv_df.head()

In [0]:
#me representa todos los cuantiles y todos los tamaños de los modelos
from utilsforecast.plotting import plot_series

#ploteo menos rango temporal para apreciar mejor las predicciones
df_plot = df.groupby("unique_id").tail(24 * 15)

plot_series(
    df_plot,
    forecasts_df=cv_df.drop(columns=["cutoff"], errors="ignore")
)

**8. Selección del mejor modelo y visualización**

In [0]:
#aquí es donde me quedo con una sola linea de forecast (q50) y transformo el resto de cuantiles en intervalos de confianza
#determino la estacionalidad de mi serie (provisional --> me gustaría que el notebook definitivo fuera lo más genérico posible)
SEASONALITY = 12

# Niveles que corresponden a los cuantiles calculados.
LEVELS = [40, 60, 80]
# Equivalencias:
#   intervalo 40% = q30-q70
#   intervalo 60% = q20-q80
#   intervalo 80% = q10-q90
LEVEL_TO_QUANTILES = {
    40: (30, 70),
    60: (20, 80),
    80: (10, 90),
}

# Intervalo que se representará en las gráficas.
PLOT_LOWER_QUANTILE = 20
PLOT_CENTRAL_QUANTILE = 50
PLOT_UPPER_QUANTILE = 80

# Número de subgráficos por fila.
N_PLOT_COLUMNS = 2


# compruebo los dataframes de entrada
required_df_columns = {"unique_id", "ds", "y"}
required_cv_columns = {"unique_id", "ds", "cutoff", "y"}

missing_df_columns = required_df_columns - set(df.columns)
missing_cv_columns = required_cv_columns - set(cv_df.columns)

if missing_df_columns:
    raise ValueError(
        f"Faltan estas columnas en df: {sorted(missing_df_columns)}"
    )

if missing_cv_columns:
    raise ValueError(
        f"Faltan estas columnas en cv_df: "
        f"{sorted(missing_cv_columns)}"
    )

# Trabajamos con copias para no modificar df ni cv_df.
df_work = df.copy()
cv_work = cv_df.copy()

# Asegurar formato de fecha.
df_work["ds"] = pd.to_datetime(df_work["ds"])
cv_work["ds"] = pd.to_datetime(cv_work["ds"])
cv_work["cutoff"] = pd.to_datetime(cv_work["cutoff"])

# Se consideran modelos las columnas que:
#   - no son unique_id, ds, cutoff o y;
#   - no son columnas de cuantiles "-q-";
#   - no son columnas de intervalos "-lo-" o "-hi-".
# ---------------------------------------------------------------------

metadata_columns = {
    "unique_id",
    "ds",
    "cutoff",
    "y",
}

all_models = [
    column
    for column in cv_work.columns
    if column not in metadata_columns
    and "-q-" not in column
    and "-lo-" not in column
    and "-hi-" not in column
]

print(f"Modelos encontrados en cv_df: {len(all_models)}")

# Normalizar cuantiles: CITRAS usa q-0.1 y el resto del código espera q-10.
for model in all_models:
    for quantile in required_quantiles:
        expected_col = f"{model}-q-{quantile}"
        decimal_col = f"{model}-q-{quantile / 100:g}"

        if expected_col not in cv_work.columns and decimal_col in cv_work.columns:
            cv_work[expected_col] = cv_work[decimal_col]

# ---------------------------------------------------------------------
# 5. CONSERVAR SOLO MODELOS CON TODOS LOS CUANTILES NECESARIOS
# ---------------------------------------------------------------------
required_quantiles = [10, 20, 30, 50, 70, 80, 90]

models_with_all_quantiles = [
    model
    for model in all_models
    if all(
        f"{model}-q-{quantile}" in cv_work.columns
        for quantile in required_quantiles
    )
]

models_without_all_quantiles = [
    model
    for model in all_models
    if model not in models_with_all_quantiles
]


if models_without_all_quantiles:
    print("\nModelos excluidos por no tener todos los cuantiles:")

    for model in models_without_all_quantiles:
        print(f"  - {model}")

if not models_with_all_quantiles:
    raise ValueError(
        "No hay ningún modelo con todos los cuantiles necesarios."
    )

# ---------------------------------------------------------------------
# 6. CREAR CV_EVAL COMPATIBLE CON UTILSFORECAST
# ---------------------------------------------------------------------
#
# Este dataframe NO debe contener columnas "-q-".
#
# Si las incluyéramos, evaluate() interpretaría, por ejemplo,
# "TimeGPT-q-10" como si fuera otro modelo.
#
# Para cada modelo, cv_eval contendrá únicamente:
#
#   modelo          = q50, predicción central
#   modelo-lo-40    = q30
#   modelo-hi-40    = q70
#   modelo-lo-60    = q20
#   modelo-hi-60    = q80
#   modelo-lo-80    = q10
#   modelo-hi-80    = q90
# ---------------------------------------------------------------------

cv_eval = cv_work[
    ["unique_id", "ds", "cutoff", "y"]
].copy()

# Para evitar avisos de fragmentación de pandas, creamos primero
# todas las columnas nuevas en un diccionario.
forecast_columns = {}

for model in models_with_all_quantiles:

    # Predicción puntual: mediana q50.
    forecast_columns[model] = cv_work[f"{model}-q-50"]

    # Intervalos requeridos por evaluate(level=[40, 60, 80]).
    for level, (lower_q, upper_q) in LEVEL_TO_QUANTILES.items():

        forecast_columns[f"{model}-lo-{level}"] = (
            cv_work[f"{model}-q-{lower_q}"]
        )

        forecast_columns[f"{model}-hi-{level}"] = (
            cv_work[f"{model}-q-{upper_q}"]
        )

cv_eval = pd.concat(
    [
        cv_eval,
        pd.DataFrame(
            forecast_columns,
            index=cv_eval.index,
        ),
    ],
    axis=1,
)

# ---------------------------------------------------------------------
# 7. VALIDAR CV_EVAL ANTES DE EVALUAR
# ---------------------------------------------------------------------

# No debe haber ninguna columna "-q-".
quantile_columns_accidentally_included = [
    column
    for column in cv_eval.columns
    if "-q-" in column
]

if quantile_columns_accidentally_included:
    raise ValueError(
        "cv_eval contiene columnas de cuantiles que no deberían estar: "
        f"{quantile_columns_accidentally_included[:10]}"
    )

# Comprobar que existen todos los intervalos.
required_interval_columns = {
    f"{model}-{bound}-{level}"
    for model in models_with_all_quantiles
    for level in LEVELS
    for bound in ["lo", "hi"]
}

missing_interval_columns = (
    required_interval_columns - set(cv_eval.columns)
)

if missing_interval_columns:
    raise ValueError(
        "Faltan columnas de intervalos: "
        f"{sorted(missing_interval_columns)[:20]}"
    )

print(
    f"cv_eval preparado correctamente con "
    f"{len(models_with_all_quantiles)} modelos."
)

print(
    "¿Contiene columnas '-q-'?:",
    any("-q-" in column for column in cv_eval.columns),
)



In [0]:
# 8. CONSTRUIR TRAIN_DF
# ---------------------------------------------------------------------
#
# MASE, MSSE, RMSSE y SPIS necesitan el histórico de entrenamiento.
#
# Se utiliza el primer cutoff de cada serie para impedir que entren
# observaciones correspondientes al periodo de validación.
# ---------------------------------------------------------------------

first_cutoff_by_series = (
    cv_eval
    .groupby("unique_id", as_index=False)["cutoff"]
    .min()
    .rename(columns={"cutoff": "first_cutoff"})
)

train_df = (
    df_work
    .merge(
        first_cutoff_by_series,
        on="unique_id",
        how="inner",
    )
)

train_df = (
    train_df.loc[
        train_df["ds"] <= train_df["first_cutoff"],
        ["unique_id", "ds", "y"],
    ]
    .copy()
)

# ---------------------------------------------------------------------
# 9. CALCULAR TODAS LAS MÉTRICAS
# ---------------------------------------------------------------------
#
# Importante:
#   - cv_eval sí tiene columnas lo/hi.
#   - cv_eval no tiene columnas q.
#   - level informa a evaluate de los niveles existentes.
# ---------------------------------------------------------------------

eval_df = evaluate(
    cv_eval.drop(columns=["cutoff"]),
    train_df=train_df,
    level=LEVELS,
    metrics=[
        # Errores absolutos y cuadráticos
        mae,
        rmse,

        # Errores porcentuales
        smape,

        # Errores escalados
        partial(
            mase,
            seasonality=SEASONALITY,
        ),

        # Métricas de inventario
        pis,
        spis,

        # Métricas probabilísticas
        coverage,
        calibration,
        winkler_score,
        scaled_crps,
    ],
)


In [0]:
# 10. CREAR TABLA DE RESULTADOS
# ---------------------------------------------------------------------
#
# evaluate() devuelve resultados por serie y métrica.
# Se calcula la media entre todas las series.
#
# Las filas serán modelos y las columnas serán métricas.
# ---------------------------------------------------------------------

scores = (
    eval_df
    .groupby("metric")
    .mean(numeric_only=True)
    .T
)
# 1. Limpiamos el nombre del eje de las columnas
scores.columns.name = None
# 2. Le damos nombre al índice para saber que son los modelos
scores.index.name = "model"
# Conservar exclusivamente los modelos reales.
scores = scores.loc[
    scores.index.intersection(models_with_all_quantiles)
].copy()
# 3. (Opcional pero recomendado) Convertimos el índice en una columna normal
# para que al hacer display() o guardarlo se vea perfecta.
scores = scores.reset_index()
if scores.empty:
    raise ValueError(
        "evaluate() no ha producido resultados para ningún modelo."
    )


# ---------------------------------------------------------------------
# 11. PREPARAR LAS MÉTRICAS PARA EL RANKING
# ---------------------------------------------------------------------

ranking_values = scores.copy()

metrics_excluded_from_ranking = []

for metric_name in list(ranking_values.columns):

    metric_text = str(metric_name).lower()

    # Sesgo, error acumulado y calibración:
    # cuanto más cerca de cero, mejor.
    if (
        metric_text == "bias"
        or metric_text == "cfe"
        or "calibration" in metric_text
    ):
        ranking_values[metric_name] = (
            ranking_values[metric_name].abs()
        )

    # Coverage:
    # cuanto más cerca de la cobertura nominal, mejor.
    elif "coverage" in metric_text:

        nominal_level = None

        for level in LEVELS:
            if str(level) in metric_text:
                nominal_level = level
                break

        if nominal_level is None:
            # Si el nombre de la métrica no indica si corresponde
            # al 40%, 60% u 80%, no puede incluirse correctamente
            # en el ranking.
            #
            # Se mantiene en scores para inspeccionarla.
            metrics_excluded_from_ranking.append(metric_name)

        else:
            coverage_values = ranking_values[metric_name]

            # Detectar si coverage usa escala 0-1 o 0-100.
            if coverage_values.abs().max() <= 1.5:
                target_coverage = nominal_level / 100
            else:
                target_coverage = nominal_level

            ranking_values[metric_name] = (
                coverage_values - target_coverage
            ).abs()


# Eliminar únicamente las coberturas cuyo nivel no pueda identificarse.
if metrics_excluded_from_ranking:

    ranking_values = ranking_values.drop(
        columns=metrics_excluded_from_ranking
    )

# ---------------------------------------------------------------------
# 12. TRATAR NAN E INFINITOS
# ---------------------------------------------------------------------
#
# MAPE puede producir infinito cuando y = 0.
#
# Una métrica se usa si tiene al menos un valor finito.
# Cuando un modelo concreto tenga NaN, quedará al final en esa métrica.
# ---------------------------------------------------------------------

ranking_values = ranking_values.replace(
    [np.inf, -np.inf],
    np.nan,
)

valid_ranking_metrics = [
    metric_name
    for metric_name in ranking_values.columns
    if ranking_values[metric_name].notna().any()
]

ranking_values = ranking_values[
    valid_ranking_metrics
].copy()

if ranking_values.empty:
    raise ValueError(
        "No hay métricas válidas para crear el ranking."
    )



In [0]:
# 13. CALCULAR RANKING GLOBAL
# ---------------------------------------------------------------------
#
# Cada modelo recibe una posición dentro de cada métrica:
#
#   1 = mejor
#   2 = segundo
#   3 = tercero
#
# rank_score es la posición media.
#
# Menor rank_score = mejor resultado global.
# Todas las métricas válidas tienen el mismo peso.
# ---------------------------------------------------------------------

ranks = ranking_values.rank(
    axis=0,
    method="average",
    ascending=True,
    na_option="bottom",
)

scores["rank_score"] = ranks.mean(axis=1)

scores = scores.sort_values(
    by="rank_score",
    ascending=True,
)


**8.1. Muestro la tabla completa con todos los modelos, las métricas estudiadas y el rank_score**

In [0]:
import os

OUTPUT_DIR = (
    "/Workspace/Users/anamaria.afonso@practicas.repsol.com/timecopilot_pruebas/mis-resultados/resultados-exog"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

import plotly.express as px
# Tabla corta y ranking global: menor rank_score es mejor.
metric_priority = ["mae", "rmse", "smape", "mase", "scaled_crps"]
metric_cols = [col for col in metric_priority if col in scores.columns]

# Si los nombres de métricas difieren, mostramos algunas de las disponibles.
if not metric_cols:
    metric_cols = [
        col for col in scores.columns
        if col not in {"model", "rank_score"}
    ][:4]

leaderboard = (
    scores.sort_values("rank_score")
    .loc[:, ["model", "rank_score"] + metric_cols]
    .head(25)
    .reset_index(drop=True)
)

display(leaderboard.round(3))

ranking_plot = (
    leaderboard.sort_values("rank_score", ascending=False)
    .copy()
)

fig = px.bar(
    ranking_plot,
    x="rank_score",
    y="model",
    orientation="h",
    title="Top 25 modelos de 49— ranking global",
    labels={"rank_score": "Rank score (menor es mejor)", "model": "Modelo"},
    text="rank_score",
)
fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.update_layout(
    template="plotly_white",
    height=max(400, 28 * len(ranking_plot)),
    showlegend=False,
    xaxis_title="Rank score (menor es mejor)",
    yaxis_title="",
)
fig.write_html(
    os.path.join(OUTPUT_DIR, "ranking_top25.html"),
    include_plotlyjs="cdn",
)
fig.show()

**8.2.Comparación visual del ultimo cutoff para los 10 mejores modelos**

In [0]:
# Comparación visual del último cutoff para los 25 mejores modelos.
import os
import re
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

OUTPUT_DIR = (
    "/Workspace/Users/anamaria.afonso@practicas.repsol.com/"
    "timecopilot_pruebas/mis-resultados/resultados-exog"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_TOP_MODELS = 25
HISTORY_POINTS = 72
LOWER_Q, CENTRAL_Q, UPPER_Q = 20, 50, 80
PLOT_SERIES = None  # Ejemplo: ["serie_1", "serie_2"]

required_base = {"unique_id", "ds", "cutoff", "y"}
missing_base = required_base - set(cv_work.columns)
if missing_base:
    raise ValueError(f"Faltan columnas en cv_work: {sorted(missing_base)}")

ranked_models = (
    scores.loc[scores["model"].isin(models_with_all_quantiles)]
    .sort_values("rank_score")
)
ranked_models = ranked_models.head(N_TOP_MODELS)["model"].tolist()

if not ranked_models:
    raise ValueError("No hay modelos disponibles para representar.")

required_forecast_cols = [
    f"{model}-q-{q}"
    for model in ranked_models
    for q in (LOWER_Q, CENTRAL_Q, UPPER_Q)
]
missing_forecast_cols = set(required_forecast_cols) - set(cv_work.columns)
if missing_forecast_cols:
    raise ValueError(
        f"Faltan columnas de cuantiles: {sorted(missing_forecast_cols)}"
    )

cv_plot = cv_work.copy()
cv_plot["ds"] = pd.to_datetime(cv_plot["ds"])
cv_plot["cutoff"] = pd.to_datetime(cv_plot["cutoff"])

# Seleccionar el cutoff más reciente de cada serie, sin mezclar folds.
latest_cutoff = cv_plot.groupby("unique_id")["cutoff"].transform("max")
cv_plot = cv_plot.loc[cv_plot["cutoff"].eq(latest_cutoff)].copy()

if PLOT_SERIES is None:
    scale_by_series = (
        df_work.groupby("unique_id")["y"]
        .apply(lambda values: values.abs().median())
        .dropna()
        .sort_values()
    )
    if scale_by_series.empty:
        raise ValueError("No hay series con valores históricos válidos.")

    positions = [0, len(scale_by_series) // 2, len(scale_by_series) - 1]
    plot_series_ids = list(dict.fromkeys(
        scale_by_series.index[position] for position in positions
    ))
else:
    plot_series_ids = list(PLOT_SERIES)

colors = px.colors.qualitative.Dark24 + px.colors.qualitative.Light24

for series_index, unique_id in enumerate(plot_series_ids, start=1):
    fold = (
        cv_plot.loc[cv_plot["unique_id"].eq(unique_id)]
        .sort_values("ds")
        .copy()
    )
    if fold.empty:
        print(f"Se omite {unique_id}: no hay predicciones para el último cutoff.")
        continue

    cutoffs = fold["cutoff"].dropna().unique()
    if len(cutoffs) != 1:
        raise ValueError(f"{unique_id}: se esperaba un único cutoff reciente.")
    cutoff = pd.Timestamp(cutoffs[0])

    if fold["ds"].duplicated().any():
        raise ValueError(f"{unique_id}: hay fechas duplicadas en el último fold.")

    forecast = fold.loc[fold["ds"] > cutoff].copy()
    if forecast.empty:
        print(f"Se omite {unique_id}: no hay horizonte de validación.")
        continue

    history = (
        df_work.loc[
            df_work["unique_id"].eq(unique_id)
            & (pd.to_datetime(df_work["ds"]) <= cutoff),
            ["ds", "y"],
        ]
        .sort_values("ds")
        .tail(HISTORY_POINTS)
    )

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=history["ds"],
        y=history["y"],
        mode="lines",
        name="Histórico de entrenamiento",
        line=dict(color="#333333", width=2),
    ))

    fig.add_trace(go.Scatter(
        x=forecast["ds"],
        y=forecast["y"],
        mode="lines+markers",
        name="Real (validación)",
        line=dict(color="#D62728", width=2, dash="dot"),
        marker=dict(size=6),
    ))

    for index, model in enumerate(ranked_models):
        color = colors[index % len(colors)]
        lower = forecast[f"{model}-q-{LOWER_Q}"]
        upper = forecast[f"{model}-q-{UPPER_Q}"]

        # Las tres trazas comparten grupo: leyenda oculta banda y predicción.
        fig.add_trace(go.Scatter(
            x=forecast["ds"],
            y=upper,
            mode="lines",
            line=dict(color=color, width=0),
            name=f"{model} límite superior",
            legendgroup=model,
            showlegend=False,
            hoverinfo="skip",
        ))

        fig.add_trace(go.Scatter(
            x=forecast["ds"],
            y=lower,
            mode="lines",
            line=dict(color=color, width=0),
            fill="tonexty",
            fillcolor=f"rgba(100, 100, 100, 0.12)",
            name=f"{model} intervalo",
            legendgroup=model,
            showlegend=False,
            hoverinfo="skip",
        ))

        fig.add_trace(go.Scatter(
            x=forecast["ds"],
            y=forecast[f"{model}-q-{CENTRAL_Q}"],
            mode="lines+markers",
            name=model,
            legendgroup=model,
            line=dict(color=color, width=2),
            marker=dict(size=4),
        ))

    fig.add_vline(
        x=cutoff,
        line_dash="dash",
        line_color="#777777",
        annotation_text="cutoff",
    )

    fig.update_layout(
        title=f"Serie {unique_id} — top {len(ranked_models)} modelos",
        template="plotly_white",
        height=650,
        hovermode="x unified",
        xaxis_title="Fecha",
        yaxis_title="Valor",
        legend=dict(
            title="Real / modelo",
            groupclick="togglegroup",
        ),
    )

    safe_id = re.sub(r"[^A-Za-z0-9._-]+", "_", str(unique_id)).strip("._")
    if not safe_id:
        safe_id = f"serie_{series_index}"

    output_path = os.path.join(
        OUTPUT_DIR,
        f"top25_serie_{series_index}_{safe_id}.html",
    )
    fig.write_html(output_path, include_plotlyjs="cdn")
    print(f"Gráfica guardada: {output_path}")

    fig.show()

**9. Visualización de resultados inspirados en  https://journals-sol.sbc.org.br/index.php/jbcs/article/view/6840/3977**

Celda 1 — Preparar resultados y resumen por clase

In [0]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Clasificación explícita para evitar inferir clases a partir del nombre.
STATISTICAL_MODELS = {"SeasonalNaive", "AutoARIMA", "AutoETS"}
ML_MODELS = {
    "AutoXGBoost",
    "AutoLGBM",
    "AutoRandomForest",
    "AutoLinearRegression",
    "AutoRidge",
    "AutoLasso",
}

def approach_of(model):
    if model in STATISTICAL_MODELS:
        return "Estadísticos"
    if model in ML_MODELS:
        return "ML"
    return "Fundacionales"

# evaluate() debe conservar el resultado por serie y métrica.
required_eval_cols = {"unique_id", "metric"}
missing_eval_cols = required_eval_cols - set(eval_df.columns)
if missing_eval_cols:
    raise ValueError(
        f"Faltan columnas en eval_df: {sorted(missing_eval_cols)}"
    )

model_cols = [
    model for model in models_with_all_quantiles
    if model in eval_df.columns
]
if not model_cols:
    raise ValueError("eval_df no contiene columnas de modelos.")

eval_long = eval_df.melt(
    id_vars=["unique_id", "metric"],
    value_vars=model_cols,
    var_name="model",
    value_name="value",
).dropna(subset=["value"])

eval_long["approach"] = eval_long["model"].map(approach_of)

# Resumen de promedios por clase. Las métricas se muestran en sus
# escalas originales; coverage/calibration no se interpretan como errores.
class_summary = (
    eval_long
    .groupby(["approach", "metric"], as_index=False)
    .agg(value=("value", "mean"))
    .pivot(index="approach", columns="metric", values="value")
)

class_summary.insert(
    0,
    "n_modelos",
    eval_long.groupby("approach")["model"].nunique()
        .reindex(class_summary.index),
)

display(class_summary.round(3))

# Tabla de mejores modelos por clase, según el rank_score existente.
if "model" not in scores.columns or "rank_score" not in scores.columns:
    raise ValueError("scores debe contener 'model' y 'rank_score'.")

ranked_models = scores.copy()
ranked_models["approach"] = ranked_models["model"].map(approach_of)

best_by_approach = (
    ranked_models
    .sort_values("rank_score")
    .groupby("approach", as_index=False)
    .first()
    .sort_values("rank_score")
)

display(
    best_by_approach[
        ["approach", "model", "rank_score"]
    ].round(3)
)

Celda 2 — Pronósticos reales frente a modelos

In [0]:
# Mostrar únicamente las predicciones y los valores reales de validación.
# Se eligen hasta dos modelos por clase, según el rank_score global.
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

MODELS_PER_APPROACH = 2
PLOT_SERIES = None  # Ejemplo: ["serie_1", "serie_2"]

top_by_approach = (
    ranked_models
    .sort_values("rank_score")
    .groupby("approach", sort=False)
    .head(MODELS_PER_APPROACH)
)
plot_models = top_by_approach["model"].tolist()

required_plot_cols = {"unique_id", "ds", "cutoff", "y"}
missing_plot_cols = required_plot_cols - set(cv_work.columns)
if missing_plot_cols:
    raise ValueError(
        f"Faltan columnas en cv_work: {sorted(missing_plot_cols)}"
    )

missing_models = [
    model for model in plot_models
    if f"{model}-q-50" not in cv_work.columns
]
if missing_models:
    raise ValueError(f"Falta q50 para estos modelos: {missing_models}")

cv_plot = cv_work.copy()
cv_plot["ds"] = pd.to_datetime(cv_plot["ds"])
cv_plot["cutoff"] = pd.to_datetime(cv_plot["cutoff"])

# Seleccionar el cutoff más reciente de cada serie para no mezclar folds.
last_cutoff = cv_plot.groupby("unique_id")["cutoff"].transform("max")
cv_plot = cv_plot.loc[cv_plot["cutoff"].eq(last_cutoff)].copy()

# Elegir series de escala baja, mediana y alta, o usar una selección manual.
if PLOT_SERIES is None:
    scale = (
        df_work.groupby("unique_id")["y"]
        .apply(lambda x: x.abs().median())
        .dropna()
        .sort_values()
    )
    if scale.empty:
        raise ValueError("No hay series con valores históricos válidos.")

    positions = [0, len(scale) // 2, len(scale) - 1]
    plot_ids = list(dict.fromkeys(scale.index[i] for i in positions))
else:
    plot_ids = list(PLOT_SERIES)

palette = px.colors.qualitative.Plotly

for unique_id in plot_ids:
    fold = (
        cv_plot.loc[cv_plot["unique_id"].eq(unique_id)]
        .sort_values("ds")
        .copy()
    )
    if fold.empty:
        print(f"Se omite {unique_id}: no hay predicciones.")
        continue

    cutoff = fold["cutoff"].max()
    forecast = fold.loc[fold["ds"] > cutoff].copy()

    if forecast.empty:
        print(f"Se omite {unique_id}: no hay periodo de validación.")
        continue

    fig = go.Figure()

    # Valores reales observados durante el periodo de validación.
    fig.add_trace(go.Scatter(
        x=forecast["ds"],
        y=forecast["y"],
        mode="lines+markers",
        name="Validación real",
        line=dict(color="black", width=2, dash="dot"),
        marker=dict(size=6),
    ))

    # Predicción central q50 de cada modelo.
    for i, model in enumerate(plot_models):
        fig.add_trace(go.Scatter(
            x=forecast["ds"],
            y=forecast[f"{model}-q-50"],
            mode="lines+markers",
            name=model,
            line=dict(color=palette[i % len(palette)], width=2),
            marker=dict(size=4),
        ))

    fig.update_layout(
        title=f"Predicciones frente a validación — serie {unique_id}",
        template="plotly_white",
        height=500,
        hovermode="x unified",
        xaxis_title="Fecha",
        yaxis_title="Valor",
        legend_title="Real / modelo",
    )
    fig.show()

Celda 3 — Comparación de clases y diagrama de diferencias críticas

In [0]:
from scipy.stats import friedmanchisquare, studentized_range

rmse_by_series = eval_long.loc[
    eval_long["metric"].astype(str).str.lower().eq("rmse")
].copy()

# Promedia los modelos de cada clase dentro de cada serie.
class_rmse = (
    rmse_by_series
    .groupby(["unique_id", "approach"])["value"]
    .mean()
    .unstack("approach")
    .dropna(axis=0, how="any")
)

if class_rmse.shape[0] < 2 or class_rmse.shape[1] < 3:
    raise ValueError(
        "Se necesitan al menos dos series completas y tres clases "
        "para calcular el diagrama."
    )

# RMSE menor es mejor: rango 1 = mejor clase en esa serie.
ranks_by_series = class_rmse.rank(
    axis=1,
    ascending=True,
    method="average",
)
mean_ranks = ranks_by_series.mean().sort_values()

statistic, p_value = friedmanchisquare(
    *[class_rmse[column].to_numpy() for column in class_rmse.columns]
)
print(
    f"Friedman: estadístico={statistic:.3f}, p={p_value:.4g}; "
    f"series={len(class_rmse)}"
)

alpha = 0.05
k = class_rmse.shape[1]
n = class_rmse.shape[0]

# Diferencia crítica de Nemenyi.
q_critical = studentized_range.ppf(1 - alpha, k, np.inf) / np.sqrt(2)
critical_difference = q_critical * np.sqrt(k * (k + 1) / (6 * n))
print(f"Diferencia crítica de Nemenyi (alpha={alpha}): {critical_difference:.3f}")

fig = go.Figure()

# El eje y se usa para separar visualmente las clases.
for y_position, (approach, rank) in enumerate(mean_ranks.items()):
    fig.add_trace(go.Scatter(
        x=[rank],
        y=[y_position],
        mode="markers+text",
        text=[f"{rank:.2f}"],
        textposition="middle right",
        name=approach,
        marker=dict(size=12),
        showlegend=False,
    ))

# Conecta los pares de clases cuya diferencia de rango no supera el CD.
approaches = list(mean_ranks.index)
for i, left in enumerate(approaches):
    for right in approaches[i + 1:]:
        left_rank = mean_ranks[left]
        right_rank = mean_ranks[right]
        if abs(left_rank - right_rank) <= critical_difference:
            y_left = approaches.index(left)
            y_right = approaches.index(right)
            fig.add_shape(
                type="line",
                x0=left_rank, x1=right_rank,
                y0=y_left, y1=y_right,
                line=dict(color="gray", width=4),
            )

fig.update_yaxes(
    tickmode="array",
    tickvals=list(range(len(approaches))),
    ticktext=approaches,
    title="Clase",
)
fig.update_xaxes(
    autorange="reversed",
    title="Rango medio (menor es mejor)",
)
fig.update_layout(
    title="Diagrama de diferencias críticas — RMSE por serie",
    template="plotly_white",
    height=350,
    showlegend=False,
)
fig.write_html(
        os.path.join(OUTPUT_DIR, "diagrama_diferencias_criticas.html"),
        include_plotlyjs="cdn",
    )
fig.show()

Celda 4 — Mapa de diferencias pareadas entre clases

In [0]:
approaches = class_rmse.columns.tolist()

difference_matrix = pd.DataFrame(
    index=approaches,
    columns=approaches,
    dtype=float,
)

for left in approaches:
    for right in approaches:
        difference_matrix.loc[left, right] = (
            class_rmse[left] - class_rmse[right]
        ).mean()

limit = np.nanmax(np.abs(difference_matrix.to_numpy()))
if not np.isfinite(limit) or limit == 0:
    limit = 1

fig = px.imshow(
    difference_matrix,
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    zmin=-limit,
    zmax=limit,
    text_auto=".3f",
    labels=dict(x="Clase comparada", y="Clase", color="Δ RMSE"),
    title="Diferencia pareada de RMSE entre clases",
)
fig.update_layout(template="plotly_white")
fig.write_html(
    os.path.join(OUTPUT_DIR, "diagrama_diferencias_criticas.html"),
    include_plotlyjs="cdn",
)
fig.show()

Celda 5 — Mejor fundacional frente al mejor método tradicional

In [0]:
# Mejor modelo fundacional y mejor modelo tradicional (estadístico o ML)
traditional = ranked_models.loc[
    ranked_models["approach"].isin(["Estadísticos", "ML"])
].sort_values("rank_score")

foundation = ranked_models.loc[
    ranked_models["approach"].eq("Fundacionales")
].sort_values("rank_score")

if traditional.empty or foundation.empty:
    raise ValueError(
        "No se encontraron modelos tradicionales y fundacionales."
    )

best_traditional = traditional.iloc[0]["model"]
best_foundation = foundation.iloc[0]["model"]

print("Mejor tradicional:", best_traditional)
print("Mejor fundacional:", best_foundation)

series_comparison = eval_long.loc[
    eval_long["model"].isin([best_traditional, best_foundation])
    & eval_long["metric"].astype(str).str.lower().isin(["rmse", "mase"])
].pivot_table(
    index=["unique_id", "metric"],
    columns="model",
    values="value",
    aggfunc="first",
)

if best_traditional not in series_comparison.columns:
    raise ValueError(f"No hay resultados para {best_traditional}.")
if best_foundation not in series_comparison.columns:
    raise ValueError(f"No hay resultados para {best_foundation}.")

display(series_comparison.round(3))

for metric in series_comparison.index.get_level_values("metric").unique():
    metric_results = series_comparison.xs(metric, level="metric").dropna()
    foundation_wins = (
        metric_results[best_foundation] < metric_results[best_traditional]
    ).sum()
    traditional_wins = (
        metric_results[best_traditional] < metric_results[best_foundation]
    ).sum()

    print(
        f"{metric}: fundacional gana en {foundation_wins} series; "
        f"tradicional gana en {traditional_wins}."
    )